# Construction de l'entrepôt, étape par étape

**Objectif :** comprendre **comment** dbt construit chaque table de l'entrepôt.
dbt fait tout en une seule requête ; ici, on refait chaque étape à la main et on regarde le résultat intermédiaire.

Le notebook lit les vues `stg_*` déjà construites par dbt dans `ctc_olist.duckdb`.

**Avant de commencer :** lancer `dbt build` depuis `ctc_olist/`.
**Avant un nouveau `dbt build` :** redémarrer le noyau du notebook (DuckDB n'accepte qu'un seul programme à la fois quand il écrit).

## Préparation

In [1]:
import duckdb

# Base construite par dbt, ouverte en lecture seule.
# Les vues stg_* lisent les CSV avec un chemin relatif à ctc_olist/ (data/raw/...) :
# file_search_path indique à DuckDB où chercher ces fichiers.
con = duckdb.connect(
    "../ctc_olist/ctc_olist.duckdb",
    read_only=True,
    config={"file_search_path": "../ctc_olist"},
)

def sql(query):
    """Exécute une requête et renvoie le résultat sous forme de tableau."""
    return con.sql(query).df()

## 1. `fct_orders` : réunir les commandes et les articles

`fct_orders` doit avoir **une ligne par commande** (sa granularité).
Le montant d'une commande vient pourtant d'une autre table : les articles.
Comment réunir les deux sans fausser les chiffres ?

### 1.1 Deux tables, deux granularités

In [2]:
# Nombre de lignes de chaque table
sql("""
    select
        (select count(*) from stg_orders)      as orders,
        (select count(*) from stg_order_items) as order_items
""")

,orders,order_items
0,99441,112650


In [3]:
# Combien d'articles par commande ?
sql("""
    with items_per_order as (
        select order_id, count(*) as items
        from stg_order_items
        group by order_id
    )
    select items, count(*) as orders
    from items_per_order
    group by items
    order by items
""")

,items,orders
0,1,88863
1,2,7516
2,3,1322
3,4,505
4,5,204
5,6,198
6,7,22
7,8,8
8,9,3
9,10,8


**À retenir :**
- `stg_orders` : **une ligne par commande** (99 441).
- `stg_order_items` : **une ligne par article** (112 650).
- La plupart des commandes ont un seul article, mais **9 803 commandes en ont plusieurs** (jusqu'à 21).

### 1.2 L'erreur classique : joindre d'abord

Que se passe-t-il si on joint directement les deux tables ?

In [4]:
# Jointure directe : combien de lignes, combien de commandes ?
sql("""
    select
        count(*)                 as rows,
        count(distinct order_id) as orders
    from stg_orders
    left join stg_order_items using (order_id)
""")

,rows,orders
0,113425,99441


In [5]:
# Une commande de 3 articles après la jointure directe
sql("""
    with one_order as (
        select order_id
        from stg_order_items
        group by order_id
        having count(*) = 3
        order by order_id
        limit 1
    )
    select
        o.order_id,
        o.order_status,
        o.estimated_delivery_date,
        i.order_item_number,
        i.item_total
    from stg_orders o
    left join stg_order_items i using (order_id)
    where o.order_id in (select order_id from one_order)
""")

,order_id,order_status,estimated_delivery_date,order_item_number,item_total
0,00143d0f86d6fbd9f9b38ab440ac16f5,delivered,2017-11-06,1,36.43
1,00143d0f86d6fbd9f9b38ab440ac16f5,delivered,2017-11-06,2,36.43
2,00143d0f86d6fbd9f9b38ab440ac16f5,delivered,2017-11-06,3,36.43


La jointure donne **113 425 lignes pour 99 441 commandes**.
Une commande de 3 articles apparaît **3 fois** : son statut et sa date estimée sont répétés sur chaque ligne.

Conséquence : tout ce qui se compte **par commande** devient faux. Exemple avec les commandes en retard :

In [6]:
# Commandes en retard : avec la jointure directe, puis sans jointure
sql("""
    select
        (select count(*)
         from stg_orders
         left join stg_order_items using (order_id)
         where cast(delivered_to_customer_at as date) > estimated_delivery_date) as late_after_join,

        (select count(*)
         from stg_orders
         where cast(delivered_to_customer_at as date) > estimated_delivery_date) as late_real
""")

,late_after_join,late_real
0,7265,6535


**À retenir :** la jointure directe compte **7 265** commandes en retard au lieu de **6 535**.
Les commandes de plusieurs articles sont comptées plusieurs fois : **730 retards de trop** (+11 %).

### 1.3 La bonne méthode : agréger d'abord, joindre ensuite

On ramène d'abord les articles à la granularité de la commande (une ligne par commande), **puis** on joint.
C'est exactement le bloc `order_items` de `models/marts/fct_orders.sql`.

In [7]:
# Étape 1 : les articles additionnés par commande
sql("""
    select
        order_id,
        sum(item_total) as order_amount
    from stg_order_items
    group by order_id
    order by order_id
    limit 5
""")

,order_id,order_amount
0,00010242fe8c5a6d1ba2dd792cb16214,72.19
1,00018f77f2f0320c557190d7a144bdd3,259.83
2,000229ec398224ef6ca0657da4fc703e,216.87
3,00024acbcdf0a6daa1e931b038114c75,25.78
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04


In [8]:
# Étape 2 : la jointure avec les commandes
sql("""
    with order_items as (
        select
            order_id,
            sum(item_total) as order_amount
        from stg_order_items
        group by order_id
    )
    select
        count(*)                                  as rows,
        count(distinct order_id)                  as orders,
        sum(coalesce(order_amount, 0))            as total_revenue,
        count(*) filter (where order_amount is null) as orders_without_items
    from stg_orders
    left join order_items using (order_id)
""")

,rows,orders,total_revenue,orders_without_items
0,99441,99441,15843553.24,775


**À retenir :**
- **99 441 lignes = 99 441 commandes** : la granularité est respectée.
- Chiffre d'affaires : **15 843 553,24 BRL**.
- **775 commandes** n'ont aucun article (indisponibles ou annulées). Le `left join` les garde, et `coalesce` met leur montant à 0.

Vérifions que dbt obtient le même résultat :

In [9]:
# Le résultat construit par dbt
sql("""
    select
        count(*)          as rows,
        sum(order_amount) as total_revenue
    from fct_orders
""")

,rows,total_revenue
0,99441,15843553.24


### En résumé

| Méthode | Lignes | Commandes en retard |
|---|---|---|
| Joindre, puis compter | 113 425 | 7 265 (faux) |
| Agréger, puis joindre | 99 441 | 6 535 |

**La règle :** avant de joindre une table à un fait, la ramener à la **granularité du fait**.
C'est l'étape 2 du cours (déterminer la granularité) : une table de faits a une seule granularité, et chaque jointure doit la respecter.

## 2. `fct_orders` : ajouter l'avis et le sentiment

Après le montant, `fct_orders` reçoit la **note** et le **sentiment** de l'avis du client.
Même question qu'en section 1 : les avis ont-ils la même granularité que les commandes ?

### 2.1 La granularité des avis

In [10]:
# Nombre d'avis et nombre de commandes qui ont un avis
sql("""
    select
        count(*)                 as reviews,
        count(distinct order_id) as orders_with_review,
        (select count(*) from stg_orders) - count(distinct order_id) as orders_without_review
    from stg_order_reviews
""")

,reviews,orders_with_review,orders_without_review
0,99224,98673,768


In [11]:
# Combien d'avis par commande ?
sql("""
    with reviews_per_order as (
        select order_id, count(*) as reviews
        from stg_order_reviews
        group by order_id
    )
    select reviews, count(*) as orders
    from reviews_per_order
    group by reviews
    order by reviews
""")

,reviews,orders
0,1,98126
1,2,543
2,3,4


**À retenir :**
- `stg_order_reviews` a **une ligne par avis** (99 224), pas une ligne par commande.
- **547 commandes ont plusieurs avis** (543 en ont 2, 4 en ont 3).
- **768 commandes n'ont aucun avis.**

Si on joint directement, on retombe dans le piège de la section 1 :

In [12]:
# Jointure directe commandes + avis
sql("""
    select
        count(*)                 as rows,
        count(distinct order_id) as orders
    from stg_orders
    left join stg_order_reviews using (order_id)
""")

,rows,orders
0,99992,99441


**99 992 lignes pour 99 441 commandes** : 551 lignes de trop.
Ici, on ne peut pas **additionner** les avis comme les articles : une note de 4 et une note de 5 ne font pas une note de 9.
Il faut **choisir** un seul avis par commande.

### 2.2 La règle : garder l'avis le plus récent

C'est une **règle métier** : le dernier avis reflète l'opinion finale du client.
`row_number()` numérote les avis de chaque commande, du plus récent (1) au plus ancien. On garde le numéro 1.

In [13]:
# Une commande avec deux avis de notes différentes
sql("""
    select
        order_id,
        review_id,
        review_score,
        answered_at,
        row_number() over (
            partition by order_id
            order by answered_at desc, review_id
        ) as rank
    from stg_order_reviews
    where order_id = '013056cfe49763c6f66bda03396c5ee3'
""")

,order_id,review_id,review_score,answered_at,rank
0,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-05 17:02:00,1
1,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-23 12:12:30,2


L'avis du 5 mars (note 4) a le rang 1 : il est gardé. L'avis du 23 février (note 5) est écarté.
`review_id` départage deux avis répondus au même moment, pour que le résultat soit toujours le même.

`qualify` filtre sur ce rang, comme un `where`, mais **après** le calcul de `row_number()` :

In [14]:
# Un seul avis par commande
sql("""
    with latest_review as (
        select *
        from stg_order_reviews
        qualify row_number() over (
            partition by order_id
            order by answered_at desc, review_id
        ) = 1
    )
    select
        count(*)                 as reviews,
        count(distinct order_id) as orders
    from latest_review
""")

,reviews,orders
0,98673,98673


**À retenir :** 98 673 avis pour 98 673 commandes : **un avis par commande**, la granularité du fait.

### 2.3 Du commentaire au sentiment

Le commentaire est du **texte libre** (donnée non structurée). On le transforme en un score, en 3 étapes.
Exemple avec un avis noté 4 :

In [15]:
# Le commentaire de l'exemple
sql("""
    select review_score, comment_message
    from stg_order_reviews
    where review_id = '00fe7ad9bd275095529b06d8d0bc3406'
""")

,review_score,comment_message
0,4,Produto bom com entrega rapida mas veio 1 cabi...


In [16]:
# Étape 1 : découper le commentaire en mots (minuscules, sans accents)
sql("""
    select unnest(regexp_split_to_array(lower(strip_accents(comment_message)), '[^a-z]+')) as word
    from stg_order_reviews
    where review_id = '00fe7ad9bd275095529b06d8d0bc3406'
""")

,word
0,produto
1,bom
2,com
3,entrega
4,rapida
5,mas
6,veio
7,cabide
8,quebrado
9,


In [17]:
# Étape 2 : garder les mots présents dans le lexique, avec leur polarité (+1 / -1)
sql("""
    with words as (
        select unnest(regexp_split_to_array(lower(strip_accents(comment_message)), '[^a-z]+')) as word
        from stg_order_reviews
        where review_id = '00fe7ad9bd275095529b06d8d0bc3406'
    )
    select words.word, sentiment_lexicon.polarity
    from words
    inner join sentiment_lexicon on words.word = sentiment_lexicon.word
""")

,word,polarity
0,bom,1
1,rapida,1
2,quebrado,-1


In [18]:
# Étape 3 : additionner les polarités, puis donner une étiquette
sql("""
    with words as (
        select unnest(regexp_split_to_array(lower(strip_accents(comment_message)), '[^a-z]+')) as word
        from stg_order_reviews
        where review_id = '00fe7ad9bd275095529b06d8d0bc3406'
    )
    select
        sum(polarity) as sentiment_score,
        case
            when sum(polarity) > 0 then 'positive'
            when sum(polarity) < 0 then 'negative'
            else 'neutral'
        end as sentiment_label
    from words
    inner join sentiment_lexicon on words.word = sentiment_lexicon.word
""")

,sentiment_score,sentiment_label
0,1.0,positive


**À retenir :** « bom » (+1) + « rapida » (+1) + « quebrado » (−1) = **+1 → positive**.
Le client a mis 4 sur 5 : le sentiment calculé est cohérent avec sa note.
La ligne vide de l'étape 1 vient du point final ; elle ne correspond à aucun mot du lexique et disparaît à l'étape 2.

C'est ce que fait `models/intermediate/int_review_sentiment.sql`, pour toutes les commandes à la fois.

### 2.4 Ajouter l'avis à `fct_orders`

`int_review_sentiment` a maintenant **une ligne par commande** : on peut le joindre sans risque.
On utilise un `left join` pour garder les commandes sans avis.

In [19]:
# Le résultat dans fct_orders
sql("""
    select
        count(*)                                           as orders,
        count(*) filter (where review_score is null)       as without_review,
        count(*) filter (where sentiment_label is null)    as without_comment
    from fct_orders
""")

,orders,without_review,without_comment
0,99441,768,58675


**À retenir :**
- **99 441 lignes** : toujours une par commande.
- **768 commandes** sans avis : `review_score` est vide (`null`).
- **58 675 commandes** sans commentaire (les 768 sans avis + 57 907 avis sans texte) : `sentiment_label` est vide.

### En résumé

| Étape | Modèle | Lignes | Granularité |
|---|---|---|---|
| Avis nettoyés | `stg_order_reviews` | 99 224 | un avis |
| Un avis par commande + sentiment | `int_review_sentiment` | 98 673 | une commande avec avis |
| Fait | `fct_orders` | 99 441 | une commande |

**Pourquoi une couche intermediate ?** Le choix de l'avis et le calcul du sentiment sont des **règles métier**.
Les isoler dans `int_review_sentiment` les rend lisibles et testables : le test `unique` sur `order_id` y prouve qu'il reste un seul avis par commande.

**Même règle qu'en section 1 :** avant de joindre, ramener la table à la granularité du fait.
Pour les articles, on **additionne** ; pour les avis, on **choisit**.

## 3. `dim_customers` : une ligne par personne

La dimension client répond à la question : **qui** a passé la commande ?
Encore une fois, tout commence par la granularité : qu'est-ce qu'**un** client ?

### 3.1 Un `customer_id` n'est pas une personne

In [20]:
# Lignes, customer_id et personnes dans stg_customers
sql("""
    select
        count(*)                           as rows,
        count(distinct customer_id)        as customer_ids,
        count(distinct customer_unique_id) as persons
    from stg_customers
""")

,rows,customer_ids,persons
0,99441,99441,96096


In [21]:
# Combien de customer_id par personne ?
sql("""
    with ids_per_person as (
        select customer_unique_id, count(*) as customer_ids
        from stg_customers
        group by customer_unique_id
    )
    select customer_ids, count(*) as persons
    from ids_per_person
    group by customer_ids
    order by customer_ids
""")

,customer_ids,persons
0,1,93099
1,2,2745
2,3,203
3,4,30
4,5,8
5,6,6
6,7,3
7,9,1
8,17,1


**À retenir :**
- Olist crée un **nouveau `customer_id` à chaque commande** : 99 441 `customer_id` pour 99 441 commandes.
- La vraie personne est `customer_unique_id` : **96 096 personnes**. Une personne a même 17 `customer_id`.

La granularité de `dim_customers` est donc **une ligne par `customer_unique_id`**.

### 3.2 Une personne, plusieurs adresses

Chaque `customer_id` porte l'adresse de livraison de **sa** commande.
Une personne qui commande depuis deux villes a donc deux adresses. Laquelle mettre dans la dimension ?

In [22]:
# Une personne avec deux adresses (une par commande)
sql("""
    select
        customers.customer_unique_id,
        orders.purchased_at,
        customers.zip_code_prefix,
        customers.city,
        customers.state
    from stg_customers customers
    inner join stg_orders orders on customers.customer_id = orders.customer_id
    where customers.customer_unique_id = '0178b244a5c281fb2ade54038dd4b161'
    order by orders.purchased_at desc
""")

,customer_unique_id,purchased_at,zip_code_prefix,city,state
0,0178b244a5c281fb2ade54038dd4b161,2018-07-28 13:13:00,14960,novo horizonte,SP
1,0178b244a5c281fb2ade54038dd4b161,2017-05-10 20:04:09,12518,guaratingueta,SP


Cette personne a commandé depuis **guaratingueta** en mai 2017, puis depuis **novo horizonte** en juillet 2018.
252 personnes sont dans ce cas.

### 3.3 La règle : garder la dernière adresse (SCD type 1)

Le cours propose plusieurs **techniques d'historisation** quand un attribut change (dimension à évolution lente, SCD) :
- **type 1** : écraser l'ancienne valeur, on ne garde que la plus récente ;
- **type 2** : garder une ligne par version, avec ses dates de validité.

Notre sujet est la satisfaction, pas les déménagements : le **type 1** suffit.

`stg_customers` n'a pas de date. Pour savoir quelle adresse est la plus récente, on la relie à la **date d'achat** de sa commande (`stg_orders`), puis on garde le rang 1 :

In [23]:
# Classer les adresses d'une personne, de la plus récente à la plus ancienne
sql("""
    select
        customers.customer_unique_id,
        orders.purchased_at,
        customers.city,
        row_number() over (
            partition by customers.customer_unique_id
            order by orders.purchased_at desc
        ) as rank
    from stg_customers customers
    inner join stg_orders orders on customers.customer_id = orders.customer_id
    where customers.customer_unique_id = '0178b244a5c281fb2ade54038dd4b161'
""")

,customer_unique_id,purchased_at,city,rank
0,0178b244a5c281fb2ade54038dd4b161,2018-07-28 13:13:00,novo horizonte,1
1,0178b244a5c281fb2ade54038dd4b161,2017-05-10 20:04:09,guaratingueta,2


In [24]:
# Appliquer la règle à tout le monde : une ligne par personne
sql("""
    with latest_address as (
        select
            customers.customer_unique_id,
            customers.city,
            customers.state
        from stg_customers customers
        inner join stg_orders orders on customers.customer_id = orders.customer_id
        qualify row_number() over (
            partition by customers.customer_unique_id
            order by orders.purchased_at desc
        ) = 1
    )
    select
        count(*)                           as rows,
        count(distinct customer_unique_id) as persons
    from latest_address
""")

,rows,persons
0,96096,96096


**À retenir :** 96 096 lignes = 96 096 personnes. C'est le bloc `latest_address` de `models/marts/dim_customers.sql`.

### 3.4 Le résultat dans `dim_customers`

In [25]:
# La personne de l'exemple dans la dimension construite par dbt
sql("""
    select *
    from dim_customers
    where customer_unique_id = '0178b244a5c281fb2ade54038dd4b161'
""")

,customer_sk,customer_unique_id,zip_code_prefix,city,state
0,e8662e7c9c6c31afbd0ef7718622bead,0178b244a5c281fb2ade54038dd4b161,14960,novo horizonte,SP


**À retenir :**
- Une seule ligne, avec la dernière adresse : **novo horizonte (SP)**.
- `customer_sk` est la **clé de substitution** : un code calculé à partir de `customer_unique_id`. C'est elle que `fct_orders` utilise pour pointer vers ce client (section 4).
- La hiérarchie géographique **ville → état** permet d'analyser par ville ou par état.

### En résumé

| Étape | Lignes | Granularité |
|---|---|---|
| `stg_customers` | 99 441 | un `customer_id` (une commande) |
| + date d'achat, rang 1 par personne | 96 096 | une personne |
| `dim_customers` | 96 096 | une personne, dernière adresse (SCD type 1) |

**Même idée qu'aux sections 1 et 2 :** avant de construire une table, fixer sa granularité, puis ramener les données à ce niveau.
Ici, on **choisit** une adresse par personne, comme on a choisi un avis par commande.

## 4. Les clés de substitution : relier le fait aux dimensions

`fct_orders` ne contient ni le nom de la ville ni le mois : seulement deux codes, `customer_sk` et `purchase_date_sk`.
Comment ces codes retrouvent-ils la bonne ligne dans `dim_customers` et `dim_date` ?

### 4.1 Ce que dbt écrit vraiment

Dans les modèles, on écrit `{{ dbt_utils.generate_surrogate_key(['customer_unique_id']) }}`.
À la compilation, dbt le remplace par ce SQL (voir `target/compiled/.../fct_orders.sql`) :

```sql
md5(cast(coalesce(cast(customer_unique_id as TEXT), '_dbt_utils_surrogate_key_null_') as TEXT))
```

`md5` est une **fonction de hachage** : elle transforme un texte en un code de 32 caractères.
Le même texte donne **toujours** le même code. Refaisons le calcul à la main pour la personne de la section 3 :

In [26]:
# Le hash calculé à la main
sql("""
    select md5(cast(coalesce(cast('0178b244a5c281fb2ade54038dd4b161' as varchar), '_dbt_utils_surrogate_key_null_') as varchar)) as hash_by_hand
""")

,hash_by_hand
0,e8662e7c9c6c31afbd0ef7718622bead


In [27]:
# La clé stockée dans dim_customers
sql("""
    select customer_unique_id, customer_sk
    from dim_customers
    where customer_unique_id = '0178b244a5c281fb2ade54038dd4b161'
""")

,customer_unique_id,customer_sk
0,0178b244a5c281fb2ade54038dd4b161,e8662e7c9c6c31afbd0ef7718622bead


**À retenir :** les deux codes sont identiques (`e8662e7c…`). Pas de table de correspondance : la clé se **recalcule**.

### 4.2 Le même calcul des deux côtés

C'est le principe qui relie tout :
- `dim_customers` calcule `customer_sk` à partir de `customer_unique_id` ;
- `fct_orders` calcule **la même chose** à partir du `customer_unique_id` de chaque commande.

Les deux codes tombent donc pareil, et la jointure fonctionne. Les commandes de notre personne :

In [28]:
# Les commandes de la personne dans fct_orders
sql("""
    select order_id, purchase_date, customer_sk
    from fct_orders
    where customer_sk = (select customer_sk from dim_customers where customer_unique_id = '0178b244a5c281fb2ade54038dd4b161')
    order by purchase_date
""")

,order_id,purchase_date,customer_sk
0,a72698cb5818851e1f19815692cf256b,2017-05-10,e8662e7c9c6c31afbd0ef7718622bead
1,94ca38619d7e64f94facac403a99682d,2018-07-28,e8662e7c9c6c31afbd0ef7718622bead


**À retenir :**
- Ses **deux commandes** pointent vers la **même** ligne de `dim_customers` : c'est bien une seule personne.
- Conséquence du SCD type 1 : la commande de mai 2017, livrée à guaratingueta, est maintenant rattachée à **novo horizonte**, la dernière adresse. L'ancienne valeur a été écrasée.

### 4.3 La même idée pour la date

`purchase_date_sk` est le hash de la date d'achat ; `date_sk` dans `dim_date` est le hash de chaque jour du calendrier.
La commande du 28 juillet 2018 retrouve donc son jour, et avec lui son mois, son trimestre et son année :

In [29]:
# Jointure fait -> dimension date pour la commande du 28 juillet 2018
sql("""
    select
        fct_orders.order_id,
        fct_orders.purchase_date_sk,
        dim_date.date_day,
        dim_date.month,
        dim_date.quarter,
        dim_date.year
    from fct_orders
    inner join dim_date on fct_orders.purchase_date_sk = dim_date.date_sk
    where fct_orders.customer_sk = (select customer_sk from dim_customers where customer_unique_id = '0178b244a5c281fb2ade54038dd4b161')
      and dim_date.date_day = '2018-07-28'
""")

,order_id,purchase_date_sk,date_day,month,quarter,year
0,94ca38619d7e64f94facac403a99682d,73fa023388893efbd216fa79c48f12c6,2018-07-28,7,3,2018


### 4.4 Toutes les commandes trouvent-elles leur client et leur jour ?

In [30]:
# Commandes reliées à une ligne de chaque dimension
sql("""
    select
        count(*)                         as orders,
        count(dim_customers.customer_sk) as with_customer,
        count(dim_date.date_sk)          as with_date
    from fct_orders
    left join dim_customers on fct_orders.customer_sk = dim_customers.customer_sk
    left join dim_date on fct_orders.purchase_date_sk = dim_date.date_sk
""")

,orders,with_customer,with_date
0,99441,99441,99441


**À retenir :** les **99 441 commandes** trouvent leur client et leur jour.
C'est exactement ce que vérifient les tests `relationships` de `_marts_models.yml`, à chaque `dbt build`.

### En résumé

**Pourquoi une clé de substitution plutôt que la clé de la source ?**
- Elle est **indépendante de la source** : si Olist changeait ses identifiants, seul le calcul changerait.
- Toutes les clés ont **le même format** (32 caractères), qu'elles viennent d'un identifiant ou d'une date.
- Le cours le souligne : les clés de substitution permettent aussi de « créer des enregistrements n'existant pas dans les systèmes sources » (comme un calendrier généré).

**Pourquoi un hash plutôt qu'un numéro 1, 2, 3… ?**
Un numéro obligerait le fait à chercher la clé dans la dimension. Avec un hash, chaque table **calcule** la clé de son côté, et le résultat est le même à chaque exécution.

## 5. Interroger l'entrepôt comme un analyste

L'entrepôt est prêt. Toutes les requêtes d'analyse suivent le même schéma en étoile :
1. partir du **fait** (`fct_orders`) ;
2. **joindre** une dimension par sa clé (`customer_sk` ou `purchase_date_sk`) ;
3. **regrouper** par un attribut de la dimension (trimestre, état…) ;
4. **agréger** les faits (somme, moyenne, taux).

### 5.1 Par trimestre (dimension date)

**Question :** comment évoluent les ventes, les retards et la note au fil de 2018 ?

In [31]:
# Ventes, taux de retard et note moyenne par trimestre de 2018
sql("""
    select
        dim_date.year,
        dim_date.quarter,
        count(*)                                     as orders,
        sum(fct_orders.order_amount)                 as revenue,
        round(count(*) filter (where fct_orders.is_late)
              / count(fct_orders.delivery_days), 2)  as late_rate,
        round(avg(fct_orders.review_score), 2)       as avg_review_score
    from fct_orders
    inner join dim_date on fct_orders.purchase_date_sk = dim_date.date_sk
    where dim_date.year = 2018
    group by dim_date.year, dim_date.quarter
    order by dim_date.year, dim_date.quarter
""")

,year,quarter,orders,revenue,late_rate,avg_review_score
0,2018,1,21208,3249337.67,0.13,3.87
1,2018,2,19979,3332156.97,0.04,4.21
2,2018,3,12820,2062202.96,0.05,4.26
3,2018,4,4,0.00,NaN,2.25


**À retenir :**
- Au 1er trimestre 2018, **13 %** des commandes livrées sont en retard, et la note moyenne tombe à **3,87**.
- Au 2e trimestre, le retard revient à **4 %**, et la note remonte à **4,21**.
- Le 4e trimestre n'a que 4 commandes : la période d'analyse s'arrête en août 2018.

Le taux de retard divise par `count(delivery_days)`, c'est-à-dire par les commandes **livrées** seulement : une commande non livrée ne peut pas être en retard.

### 5.2 Par état (dimension client)

**Question :** où livre-t-on le plus en retard ?

In [32]:
# Les 5 états avec le plus fort taux de retard
sql("""
    select
        dim_customers.state,
        count(*)                                     as orders,
        round(count(*) filter (where fct_orders.is_late)
              / count(fct_orders.delivery_days), 2)  as late_rate,
        round(avg(fct_orders.review_score), 2)       as avg_review_score
    from fct_orders
    inner join dim_customers on fct_orders.customer_sk = dim_customers.customer_sk
    group by dim_customers.state
    order by late_rate desc
    limit 5
""")

,state,orders,late_rate,avg_review_score
0,AL,412,0.21,3.76
1,MA,750,0.17,3.76
2,SE,351,0.15,3.81
3,PI,495,0.14,3.92
4,CE,1339,0.14,3.86


**À retenir :** les états du Nordeste (AL, MA, SE, PI, CE) ont **14 à 21 %** de retard, et des notes sous 4.
La hiérarchie ville → état de `dim_customers` rend cette analyse possible.

### 5.3 La question du sujet : le retard pèse-t-il sur la satisfaction ?

In [33]:
# Note moyenne des commandes livrées, à temps ou en retard
sql("""
    select
        case when is_late then 'late' else 'on_time' end as delivery,
        count(*)                                         as orders,
        round(avg(review_score), 2)                      as avg_review_score
    from fct_orders
    where delivery_days is not null
    group by delivery
""")

,delivery,orders,avg_review_score
0,late,6535,2.27
1,on_time,89941,4.29


**À retenir :** livrées à temps, les commandes ont une note de **4,29** ; en retard, de **2,27**.
Le retard de livraison est fortement lié à une mauvaise note.

### 5.4 Le lien avec la couche sémantique

Les requêtes de 5.1 demandent d'écrire la jointure, le regroupement et le bon dénominateur du taux de retard.
La couche sémantique (MetricFlow) fait ce travail à notre place. Depuis `ctc_olist/`, dans un terminal :

```bash
mf query --metrics total_revenue,late_delivery_rate,average_review_score --group-by metric_time__quarter --order metric_time__quarter --decimals 2
```

Pour le 1er trimestre 2018, MetricFlow donne **3 249 337,67 BRL**, **0,13** et **3,87** : les mêmes chiffres qu'en 5.1.
La différence : la règle (« diviser par les commandes livrées ») est écrite **une seule fois**, dans `_semantic_models.yml`, et chaque utilisateur obtient le même résultat.

## Conclusion

| Section | Problème | Solution | Notion du cours |
|---|---|---|---|
| 1 | plusieurs articles par commande | **additionner** les articles, puis joindre | granularité d'un fait |
| 2 | plusieurs avis par commande | **choisir** le plus récent, puis joindre | règle métier (couche intermediate) |
| 3 | plusieurs `customer_id` et adresses par personne | **choisir** la dernière adresse | SCD type 1 |
| 4 | relier le fait aux dimensions | **recalculer** la même clé des deux côtés | clé de substitution |
| 5 | répondre à une question d'affaires | fait + dimension + regroupement | schéma en étoile, couche sémantique |

**Le fil conducteur :** à chaque étape, on fixe d'abord la **granularité** de la table, puis on ramène les données à ce niveau avant de les joindre.

In [34]:
# Fermer la connexion (à faire avant un dbt build)
con.close()